In [113]:
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import __version__ as sklearn_version
from sklearn.decomposition import PCA
from sklearn.preprocessing import scale
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
import datetime

from library.sb_utils import save_file

In [114]:
loan_data = pd.read_csv('data/loan_eligibility_cleaned.csv')
loan_data.head().T

,0,1,2,3,4
Gender,Female,Male,Male,Male,Male
Married,No,Yes,No,Yes,Yes
Dependents,0,2,0,0,1
Education,Graduate,Graduate,Not Graduate,Graduate,Graduate
Self_Employed,No,No,No,Yes,No
Applicant_Income,2378,1299,3620,3459,5468
Coapplicant_Income,0.0,1086.0,0.0,0.0,1032.0
Loan_Amount,9,17,25,25,26
Loan_Amount_Term,360,120,120,120,360
Credit_History,1,1,1,1,1


In [115]:
loan_data.dtypes

Gender                   object
Married                  object
Dependents                int64
Education                object
Self_Employed            object
Applicant_Income          int64
Coapplicant_Income      float64
Loan_Amount               int64
Loan_Amount_Term          int64
Credit_History            int64
Property_Area            object
Loan_Status              object
Total_Income            float64
Full_Loan_Amount          int64
Monthly_Payment         float64
Debt_To_Income_Ratio    float64
dtype: object

In [116]:
loan_data.columns

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'Applicant_Income', 'Coapplicant_Income', 'Loan_Amount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status',
       'Total_Income', 'Full_Loan_Amount', 'Monthly_Payment',
       'Debt_To_Income_Ratio'],
      dtype='object')

In [118]:
#dummy = pd.get_dummies(loan_data, columns=['Gender', 'Married', 'Education', 'Self_Employed','Property_Area', 'Loan_Status'], prefix='N')
dummy = pd.get_dummies(loan_data, columns=['Property_Area'], prefix='PA', dtype=int)
dummy.head()

,Gender,Married,Dependents,Education,Self_Employed,Applicant_Income,Coapplicant_Income,Loan_Amount,Loan_Amount_Term,Credit_History,Loan_Status,Total_Income,Full_Loan_Amount,Monthly_Payment,Debt_To_Income_Ratio,PA_Rural,PA_Semiurban,PA_Urban
0,Female,No,0,Graduate,No,2378,0.0,9,360,1,N,2378.0,9000,25.00,1.05,0,0,1
1,Male,Yes,2,Graduate,No,1299,1086.0,17,120,1,Y,2385.0,17000,141.67,5.94,0,0,1
2,Male,No,0,Not Graduate,No,3620,0.0,25,120,1,Y,3620.0,25000,208.33,5.75,0,1,0
3,Male,Yes,0,Graduate,Yes,3459,0.0,25,120,1,Y,3459.0,25000,208.33,6.02,0,1,0
4,Male,Yes,1,Graduate,No,5468,1032.0,26,360,1,Y,6500.0,26000,72.22,1.11,0,1,0


In [119]:
#yes or no variables
gender_mapping = {'Male': 1, 'Female': 0}
yes_no_mapping = {'Yes': 1, 'No': 0}
edu_mapping = {'Graduate': 1, 'Not Graduate': 0}
loan_status_mapping = {'Y': 1, 'N': 0}

dummy['Gender_N'] = dummy['Gender'].map(gender_mapping)
dummy['Married_N'] = dummy['Married'].map(yes_no_mapping)
dummy['Self_Employed_N'] = dummy['Self_Employed'].map(yes_no_mapping)
dummy['Education_N'] = dummy['Education'].map(edu_mapping)
dummy['Loan_Status_N'] = dummy['Loan_Status'].map(loan_status_mapping)

In [120]:
dummy.head()

,Gender,Married,Dependents,Education,Self_Employed,Applicant_Income,Coapplicant_Income,Loan_Amount,Loan_Amount_Term,Credit_History,...,Monthly_Payment,Debt_To_Income_Ratio,PA_Rural,PA_Semiurban,PA_Urban,Gender_N,Married_N,Self_Employed_N,Education_N,Loan_Status_N
0,Female,No,0,Graduate,No,2378,0.0,9,360,1,...,25.00,1.05,0,0,1,0,0,0,1,0
1,Male,Yes,2,Graduate,No,1299,1086.0,17,120,1,...,141.67,5.94,0,0,1,1,1,0,1,1
2,Male,No,0,Not Graduate,No,3620,0.0,25,120,1,...,208.33,5.75,0,1,0,1,0,0,0,1
3,Male,Yes,0,Graduate,Yes,3459,0.0,25,120,1,...,208.33,6.02,0,1,0,1,1,1,1,1
4,Male,Yes,1,Graduate,No,5468,1032.0,26,360,1,...,72.22,1.11,0,1,0,1,1,0,1,1


In [121]:
dummy.columns

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'Applicant_Income', 'Coapplicant_Income', 'Loan_Amount',
       'Loan_Amount_Term', 'Credit_History', 'Loan_Status', 'Total_Income',
       'Full_Loan_Amount', 'Monthly_Payment', 'Debt_To_Income_Ratio',
       'PA_Rural', 'PA_Semiurban', 'PA_Urban', 'Gender_N', 'Married_N',
       'Self_Employed_N', 'Education_N', 'Loan_Status_N'],
      dtype='object')

In [122]:
dummy.dtypes

Gender                   object
Married                  object
Dependents                int64
Education                object
Self_Employed            object
Applicant_Income          int64
Coapplicant_Income      float64
Loan_Amount               int64
Loan_Amount_Term          int64
Credit_History            int64
Loan_Status              object
Total_Income            float64
Full_Loan_Amount          int64
Monthly_Payment         float64
Debt_To_Income_Ratio    float64
PA_Rural                  int64
PA_Semiurban              int64
PA_Urban                  int64
Gender_N                  int64
Married_N                 int64
Self_Employed_N           int64
Education_N               int64
Loan_Status_N             int64
dtype: object

In [123]:
df = dummy.select_dtypes(include=['int', 'float'])
df.columns

Index(['Dependents', 'Applicant_Income', 'Coapplicant_Income', 'Loan_Amount',
       'Loan_Amount_Term', 'Credit_History', 'Total_Income',
       'Full_Loan_Amount', 'Monthly_Payment', 'Debt_To_Income_Ratio',
       'PA_Rural', 'PA_Semiurban', 'PA_Urban', 'Gender_N', 'Married_N',
       'Self_Employed_N', 'Education_N', 'Loan_Status_N'],
      dtype='object')

In [124]:
df.dtypes

Dependents                int64
Applicant_Income          int64
Coapplicant_Income      float64
Loan_Amount               int64
Loan_Amount_Term          int64
Credit_History            int64
Total_Income            float64
Full_Loan_Amount          int64
Monthly_Payment         float64
Debt_To_Income_Ratio    float64
PA_Rural                  int64
PA_Semiurban              int64
PA_Urban                  int64
Gender_N                  int64
Married_N                 int64
Self_Employed_N           int64
Education_N               int64
Loan_Status_N             int64
dtype: object

In [125]:
dummy.shape

(613, 23)

In [126]:
df.shape

(613, 18)

In [127]:
df.head()

,Dependents,Applicant_Income,Coapplicant_Income,Loan_Amount,Loan_Amount_Term,Credit_History,Total_Income,Full_Loan_Amount,Monthly_Payment,Debt_To_Income_Ratio,PA_Rural,PA_Semiurban,PA_Urban,Gender_N,Married_N,Self_Employed_N,Education_N,Loan_Status_N
0,0,2378,0.0,9,360,1,2378.0,9000,25.00,1.05,0,0,1,0,0,0,1,0
1,2,1299,1086.0,17,120,1,2385.0,17000,141.67,5.94,0,0,1,1,1,0,1,1
2,0,3620,0.0,25,120,1,3620.0,25000,208.33,5.75,0,1,0,1,0,0,0,1
3,0,3459,0.0,25,120,1,3459.0,25000,208.33,6.02,0,1,0,1,1,1,1,1
4,1,5468,1032.0,26,360,1,6500.0,26000,72.22,1.11,0,1,0,1,1,0,1,1


In [128]:
df.describe()

,Dependents,Applicant_Income,Coapplicant_Income,Loan_Amount,Loan_Amount_Term,Credit_History,Total_Income,Full_Loan_Amount,Monthly_Payment,Debt_To_Income_Ratio,PA_Rural,PA_Semiurban,PA_Urban,Gender_N,Married_N,Self_Employed_N,Education_N,Loan_Status_N
count,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000
mean,0.858075,5404.729201,1619.229886,142.073409,339.425775,0.851550,7023.959086,142073.409462,456.179804,7.496754,0.292007,0.380098,0.327896,0.812398,0.649266,0.148450,0.781403,0.686786
std,1.217152,6113.949574,2928.211386,87.145169,68.508550,0.355836,6463.911930,87145.169401,377.445482,6.580326,0.455057,0.485807,0.469830,0.390713,0.477590,0.355836,0.413632,0.464179
min,0.000000,150.000000,0.000000,9.000000,36.000000,0.000000,1442.000000,9000.000000,25.000000,0.340000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,2876.000000,0.000000,98.000000,360.000000,1.000000,4166.000000,98000.000000,277.780000,5.440000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000
50%,0.000000,3812.000000,1167.000000,125.000000,360.000000,1.000000,5416.000000,125000.000000,361.110000,6.860000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000
75%,2.000000,5800.000000,2283.000000,165.000000,360.000000,1.000000,7535.000000,165000.000000,513.890000,8.170000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000
max,4.000000,81000.000000,41667.000000,700.000000,480.000000,1.000000,81000.000000,700000.000000,4305.560000,123.690000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [129]:
len(df) * .7, len(df) * .3

(429.09999999999997, 183.9)

In [130]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns='Loan_Status_N'), 
                                                    df.Loan_Status_N, test_size=0.3, 
                                                    random_state=47)

In [131]:
#X_train.shape, X_test.shape
X_train.head()

,Dependents,Applicant_Income,Coapplicant_Income,Loan_Amount,Loan_Amount_Term,Credit_History,Total_Income,Full_Loan_Amount,Monthly_Payment,Debt_To_Income_Ratio,PA_Rural,PA_Semiurban,PA_Urban,Gender_N,Married_N,Self_Employed_N,Education_N
393,0,4095,3447.0,151,360,1,7542.0,151000,419.44,5.56,1,0,0,1,0,0,1
102,0,2483,2466.0,90,180,0,4949.0,90000,500.00,10.10,1,0,0,1,1,0,1
518,0,8566,0.0,210,360,1,8566.0,210000,583.33,6.81,0,0,1,1,0,0,1
472,0,3707,3166.0,182,360,1,6873.0,182000,505.56,7.36,1,0,0,1,1,0,1
253,0,4191,0.0,120,360,1,4191.0,120000,333.33,7.95,1,0,0,1,0,0,1


In [132]:
#y_train.shape, y_test.shape
y_train.head()

393    1
102    1
518    1
472    1
253    1
Name: Loan_Status_N, dtype: int64